# LightGBM + Ensemble Approach

Flatten wearable time-series into summary statistics, combine with survey features, train LightGBM.

Three evaluation levels:
- **4-class:** healthy / prediabetes / oral_med / insulin
- **3-class:** healthy / prediabetes / diabetic (merge oral+insulin)
- **2-class:** healthy vs not_healthy

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from pathlib import Path

from config import (RANDOM_STATE, N_FOLDS, LABEL_MAP_4_TO_3, LABEL_MAP_4_TO_2,
                    CLASS_NAMES_4, CLASS_NAMES_3, CLASS_NAMES_2,
                    DAILY_FEATURE_NAMES, set_seed)

FEATURES_DIR = Path('../outputs/features')
set_seed()
print('Ready')

Ready


## Step 1: Load Raw Arrays

In [2]:
X_orig = np.load(FEATURES_DIR / 'X.npy')          # (1586, 14, 22)
X_c    = np.load(FEATURES_DIR / 'X_option_c.npy')  # (1586, 14, 19)
X_survey = np.load(FEATURES_DIR / 'X_survey.npy')  # (1586, 27)
y      = np.load(FEATURES_DIR / 'y.npy')           # (1586,)
lengths = np.load(FEATURES_DIR / 'lengths.npy')    # (1586,)

X_wear = np.concatenate([X_orig, X_c], axis=2)     # (1586, 14, 41)
N, SEQ_LEN, NF = X_wear.shape

print(f'Wearable: {X_wear.shape}  Survey: {X_survey.shape}  Labels: {np.bincount(y)}')

Wearable: (1586, 14, 41)  Survey: (1586, 27)  Labels: [560 410 446 170]


## Step 2: Flatten Wearable Features

Per feature across 14 days: mean, std, min, max, slope, last-first, days_valid

In [3]:
def flatten_temporal(X_3d, lengths):
    """Convert (N, T, F) temporal array to (N, F*7) flattened summary stats."""
    N, T, F = X_3d.shape
    stats = []
    stat_names = []
    
    for f in range(F):
        col = X_3d[:, :, f]  # (N, T)
        
        col_mean = np.nanmean(col, axis=1)
        col_std  = np.nanstd(col, axis=1)
        col_min  = np.nanmin(col, axis=1)
        col_max  = np.nanmax(col, axis=1)
        col_range = col_max - col_min
        
        # Slope: linear trend over valid days
        col_slope = np.zeros(N)
        for i in range(N):
            valid = col[i, :int(lengths[i])]
            valid = valid[~np.isnan(valid)]
            if len(valid) >= 2:
                x = np.arange(len(valid))
                col_slope[i] = np.polyfit(x, valid, 1)[0]
            else:
                col_slope[i] = 0.0
        
        # Last - First difference
        col_diff = np.zeros(N)
        for i in range(N):
            valid = col[i, :int(lengths[i])]
            valid = valid[~np.isnan(valid)]
            if len(valid) >= 2:
                col_diff[i] = valid[-1] - valid[0]
        
        # Days with valid data
        col_valid = np.sum(~np.isnan(col), axis=1).astype(float)
        
        stats.extend([col_mean, col_std, col_min, col_max, col_range, col_slope, col_diff])
        fname = f'f{f}'
        stat_names.extend([f'{fname}_mean', f'{fname}_std', f'{fname}_min', f'{fname}_max',
                           f'{fname}_range', f'{fname}_slope', f'{fname}_diff'])
    
    # Also add lengths as a feature
    stats.append(lengths.astype(float))
    stat_names.append('seq_length')
    
    X_flat = np.column_stack(stats)
    return X_flat, stat_names

X_flat, flat_names = flatten_temporal(X_wear, lengths)
print(f'Flattened wearable: {X_flat.shape} ({len(flat_names)} features)')

# Combine with survey
SURVEY_NAMES = [
    'age', 'education_years', 'smoking_ever', 'smoking_now', 'alcohol_ever',
    'cesd_score', 'sleep_restless', 'paid_score', 'sleeping_pills',
    'diet_score', 'diet_fast_food', 'diet_beans', 'diet_regular_food',
    'diet_desserts', 'diet_fats', 'fh_diabetes_parent', 'fh_diabetes_sibling',
    'has_hypertension', 'has_obesity', 'has_high_cholesterol',
    'has_heart_attack', 'has_stroke', 'has_kidney_problems', 'has_circulation',
    'vision_difficulty', 'food_insecurity_1', 'food_insecurity_2'
]

X_all = np.concatenate([X_flat, X_survey], axis=1)
all_names = flat_names + SURVEY_NAMES
print(f'Total features: {X_all.shape[1]} ({len(flat_names)} wearable + {X_survey.shape[1]} survey)')

C:\Users\hp\AppData\Local\Temp\ipykernel_3076\4077030707.py:10: RuntimeWarning: Mean of empty slice
  col_mean = np.nanmean(col, axis=1)
C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\hp\AppData\Local\Temp\ipykernel_3076\4077030707.py:12: RuntimeWarning: All-NaN slice encountered
  col_min  = np.nanmin(col, axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_3076\4077030707.py:13: RuntimeWarning: All-NaN slice encountered
  col_max  = np.nanmax(col, axis=1)


Flattened wearable: (1586, 288) (288 features)
Total features: 315 (288 wearable + 27 survey)


## Step 3: LightGBM — 4-Class

In [4]:
def compute_all_metrics(y4, proba4, label=''):
    pred = np.argmax(proba4, axis=1)
    acc = accuracy_score(y4, pred)
    auc4 = roc_auc_score(y4, proba4, multi_class='ovr', average='macro')
    
    # 3-class
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y4])
    p3 = np.zeros((len(y4), 3))
    p3[:, 0] = proba4[:, 0]; p3[:, 1] = proba4[:, 1]
    p3[:, 2] = proba4[:, 2] + proba4[:, 3]
    auc3 = roc_auc_score(y3, p3, multi_class='ovr', average='macro')
    
    # 2-class
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y4])
    auc2 = roc_auc_score(y2, proba4[:, 1] + proba4[:, 2] + proba4[:, 3])
    
    print(f'\n  {label}')
    print(f'  acc={acc*100:.1f}%  4-AUC={auc4:.4f}  3-AUC={auc3:.4f}  2-AUC={auc2:.4f}')
    print(classification_report(y4, pred, target_names=CLASS_NAMES_4, digits=3))
    return acc, auc4, auc3, auc2


def lgb_cv_4class(X, y, feature_names=None, label='LGB-4class'):
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(y), 4))
    importances = np.zeros(X.shape[1])
    
    params = {
        'objective': 'multiclass',
        'num_class': 4,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'n_estimators': 2000,
        'learning_rate': 0.02,
        'max_depth': 6,
        'num_leaves': 31,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': RANDOM_STATE,
        'verbose': -1,
        'is_unbalance': True,
    }
    
    print(f'\n[{label}] {X.shape[1]} features, {N_FOLDS}-fold CV')
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X[tr], y[tr],
            eval_set=[(X[va], y[va])],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        oof[va] = model.predict_proba(X[va])
        importances += model.feature_importances_
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    
    importances /= N_FOLDS
    acc, auc4, auc3, auc2 = compute_all_metrics(y, oof, label)
    
    # Top features
    if feature_names is not None:
        idx = np.argsort(importances)[::-1][:20]
        print('  Top-20 features:')
        for i in idx:
            print(f'    {feature_names[i]:30s}  {importances[i]:.0f}')
    
    return oof, acc, auc4, auc3, auc2

oof_lgb4, acc4, auc4_4, auc3_4, auc2_4 = lgb_cv_4class(
    X_all, y, feature_names=all_names, label='LGB-4class')
np.save(FEATURES_DIR / 'oof_lgb_4class.npy', oof_lgb4)


[LGB-4class] 315 features, 5-fold CV


  Fold 1: 126 rounds


  Fold 2: 69 rounds


  Fold 3: 65 rounds


  Fold 4: 107 rounds


  Fold 5: 89 rounds

  LGB-4class
  acc=46.3%  4-AUC=0.7024  3-AUC=0.7171  2-AUC=0.7658
              precision    recall  f1-score   support

     healthy      0.553     0.680     0.610       560
 prediabetes      0.339     0.271     0.301       410
    oral_med      0.433     0.527     0.475       446
     insulin      0.296     0.047     0.081       170

    accuracy                          0.463      1586
   macro avg      0.405     0.381     0.367      1586
weighted avg      0.436     0.463     0.436      1586

  Top-20 features:
    paid_score                      172
    has_hypertension                132
    fh_diabetes_parent              96
    f27_min                         93
    f29_mean                        90
    f16_mean                        87
    f1_mean                         87
    f4_mean                         78
    f1_max                          77
    education_years                 73
    f13_min                         72
    f8_mean                

## Step 4: LightGBM — 3-Class (dedicated model)

In [5]:
def lgb_cv_3class(X, y4, feature_names=None, label='LGB-3class'):
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y4])
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(y3), 3))
    
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'n_estimators': 2000,
        'learning_rate': 0.02,
        'max_depth': 6,
        'num_leaves': 31,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': RANDOM_STATE,
        'verbose': -1,
        'is_unbalance': True,
    }
    
    print(f'\n[{label}] {X.shape[1]} features, {N_FOLDS}-fold CV')
    for fold, (tr, va) in enumerate(skf.split(X, y3)):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X[tr], y3[tr],
            eval_set=[(X[va], y3[va])],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        oof[va] = model.predict_proba(X[va])
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    
    auc3 = roc_auc_score(y3, oof, multi_class='ovr', average='macro')
    pred = np.argmax(oof, axis=1)
    acc = accuracy_score(y3, pred)
    print(f'\n  {label}: acc={acc*100:.1f}%  3-AUC={auc3:.4f}')
    print(classification_report(y3, pred, target_names=CLASS_NAMES_3, digits=3))
    return oof, auc3

oof_lgb3, auc3_dedicated = lgb_cv_3class(X_all, y, feature_names=all_names, label='LGB-3class')
np.save(FEATURES_DIR / 'oof_lgb_3class.npy', oof_lgb3)


[LGB-3class] 315 features, 5-fold CV


  Fold 1: 122 rounds


  Fold 2: 136 rounds


  Fold 3: 121 rounds


  Fold 4: 169 rounds


  Fold 5: 131 rounds

  LGB-3class: acc=58.4%  3-AUC=0.7321
              precision    recall  f1-score   support

     healthy      0.629     0.650     0.639       560
 prediabetes      0.420     0.237     0.303       410
    diabetic      0.601     0.756     0.670       616

    accuracy                          0.584      1586
   macro avg      0.550     0.548     0.537      1586
weighted avg      0.564     0.584     0.564      1586



## Step 5: LightGBM — 2-Class (dedicated binary model)

In [6]:
def lgb_cv_2class(X, y4, feature_names=None, label='LGB-2class'):
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y4])
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y2))
    importances = np.zeros(X.shape[1])
    
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'n_estimators': 2000,
        'learning_rate': 0.02,
        'max_depth': 6,
        'num_leaves': 31,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': RANDOM_STATE,
        'verbose': -1,
        'is_unbalance': True,
    }
    
    print(f'\n[{label}] {X.shape[1]} features, {N_FOLDS}-fold CV')
    for fold, (tr, va) in enumerate(skf.split(X, y2)):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X[tr], y2[tr],
            eval_set=[(X[va], y2[va])],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        oof[va] = model.predict_proba(X[va])[:, 1]
        importances += model.feature_importances_
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    
    auc2 = roc_auc_score(y2, oof)
    pred = (oof > 0.5).astype(int)
    acc = accuracy_score(y2, pred)
    print(f'\n  {label}: acc={acc*100:.1f}%  2-AUC={auc2:.4f}')
    print(classification_report(y2, pred, target_names=CLASS_NAMES_2, digits=3))
    
    if feature_names is not None:
        importances /= N_FOLDS
        idx = np.argsort(importances)[::-1][:20]
        print('  Top-20 features (binary):')
        for i in idx:
            print(f'    {feature_names[i]:30s}  {importances[i]:.0f}')
    
    return oof, auc2

oof_lgb2, auc2_dedicated = lgb_cv_2class(X_all, y, feature_names=all_names, label='LGB-2class')
np.save(FEATURES_DIR / 'oof_lgb_2class.npy', oof_lgb2)


[LGB-2class] 315 features, 5-fold CV


  Fold 1: 231 rounds


  Fold 2: 230 rounds


  Fold 3: 114 rounds


  Fold 4: 90 rounds


  Fold 5: 249 rounds

  LGB-2class: acc=74.0%  2-AUC=0.7718
              precision    recall  f1-score   support

     healthy      0.657     0.548     0.598       560
 not_healthy      0.774     0.844     0.807      1026

    accuracy                          0.740      1586
   macro avg      0.716     0.696     0.703      1586
weighted avg      0.733     0.740     0.733      1586

  Top-20 features (binary):
    paid_score                      122
    has_hypertension                90
    fh_diabetes_parent              62
    f16_mean                        61
    f27_min                         54
    fh_diabetes_sibling             53
    f3_std                          52
    f13_min                         50
    f28_slope                       49
    has_obesity                     46
    f35_mean                        44
    f8_mean                         43
    f8_max                          39
    f31_min                         37
    f1_mean                         37

## Step 6: Ensemble — LightGBM + Hybrid LSTM

In [7]:
# Load hybrid LSTM OOF (best was T=2, a=0.5)
hybrid_oof = np.load(FEATURES_DIR / 'oof_hybrid_T2_a0.5.npy')
# Also load wearable-only distilled OOF
wearable_oof = np.load(FEATURES_DIR / 'oof_T2_a0.3.npy')

print('=== 4-CLASS ENSEMBLES ===')
print('\nLGB + Hybrid LSTM:')
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens = w * oof_lgb4 + (1 - w) * hybrid_oof
    compute_all_metrics(y, ens, f'Ens LGB×{w:.1f} + Hybrid×{1-w:.1f}')

print('\n\nLGB + Hybrid + Wearable-only (3-way):')
for wl, wh, ww in [(0.4, 0.4, 0.2), (0.5, 0.3, 0.2), (0.5, 0.4, 0.1), (0.6, 0.3, 0.1)]:
    ens = wl * oof_lgb4 + wh * hybrid_oof + ww * wearable_oof
    compute_all_metrics(y, ens, f'Ens LGB×{wl} + Hyb×{wh} + Wear×{ww}')

=== 4-CLASS ENSEMBLES ===

LGB + Hybrid LSTM:

  Ens LGB×0.3 + Hybrid×0.7
  acc=45.8%  4-AUC=0.7320  3-AUC=0.7376  2-AUC=0.7781
              precision    recall  f1-score   support

     healthy      0.587     0.675     0.628       560
 prediabetes      0.361     0.256     0.300       410
    oral_med      0.426     0.372     0.397       446
     insulin      0.295     0.453     0.357       170

    accuracy                          0.458      1586
   macro avg      0.417     0.439     0.420      1586
weighted avg      0.452     0.458     0.449      1586


  Ens LGB×0.4 + Hybrid×0.6
  acc=47.0%  4-AUC=0.7333  3-AUC=0.7394  2-AUC=0.7823
              precision    recall  f1-score   support

     healthy      0.585     0.680     0.629       560
 prediabetes      0.365     0.268     0.309       410
    oral_med      0.438     0.415     0.426       446
     insulin      0.325     0.406     0.361       170

    accuracy                          0.470      1586
   macro avg      0.429     0


  Ens LGB×0.6 + Hyb×0.3 + Wear×0.1
  acc=46.7%  4-AUC=0.7294  3-AUC=0.7363  2-AUC=0.7815
              precision    recall  f1-score   support

     healthy      0.574     0.680     0.623       560
 prediabetes      0.339     0.259     0.293       410
    oral_med      0.441     0.493     0.466       446
     insulin      0.309     0.200     0.243       170

    accuracy                          0.467      1586
   macro avg      0.416     0.408     0.406      1586
weighted avg      0.447     0.467     0.453      1586



## Step 7: Ablation — Wearable-only vs Survey-only vs Combined

In [8]:
print('=== ABLATION: Feature Source ===')
print('\n--- Wearable features only ---')
oof_w, _, _, _, _ = lgb_cv_4class(X_flat, y, flat_names, 'LGB-wearable-only')

print('\n--- Survey features only ---')
oof_s, _, _, _, _ = lgb_cv_4class(X_survey, y, SURVEY_NAMES, 'LGB-survey-only')

print('\n--- Combined (already computed above) ---')
compute_all_metrics(y, oof_lgb4, 'LGB-combined')

=== ABLATION: Feature Source ===

--- Wearable features only ---

[LGB-wearable-only] 288 features, 5-fold CV


  Fold 1: 71 rounds


  Fold 2: 48 rounds


  Fold 3: 48 rounds


  Fold 4: 68 rounds


...

  Fold 3: 95 rounds


  Fold 4: 91 rounds


  Fold 5: 66 rounds

  LGB-survey-only
  acc=46.1%  4-AUC=0.6963  3-AUC=0.7077  2-AUC=0.7649
              precision    recall  f1-score   support

     healthy      0.565     0.671     0.613       560
 prediabetes      0.313     0.227     0.263       410
    oral_med      0.428     0.552     0.482       446
     insulin      0.333     0.094     0.147       170

    accuracy                          0.461      1586
   macro avg      0.410     0.386     0.376      1586
weighted avg      0.436     0.461     0.436      1586

  Top-20 features:
    age                             1288
    paid_score                      989
    cesd_score                      931
    education_years                 874
    diet_score                      525
    fh_diabetes_parent              459
    fh_diabetes_sibling             429
    has_obesity                     403
    diet_fast_food                  347
    sleep_restless                  338
    has_hypertension                331
    diet_fat


  LGB-combined
  acc=46.3%  4-AUC=0.7024  3-AUC=0.7171  2-AUC=0.7658
              precision    recall  f1-score   support

     healthy      0.553     0.680     0.610       560
 prediabetes      0.339     0.271     0.301       410
    oral_med      0.433     0.527     0.475       446
     insulin      0.296     0.047     0.081       170

    accuracy                          0.463      1586
   macro avg      0.405     0.381     0.367      1586
weighted avg      0.436     0.463     0.436      1586



(0.4634300126103405,
 0.7023562066015765,
 0.7170554605169005,
 0.7657894736842106)

## Final Summary

In [9]:
print(f"{'Model':<50} {'Acc':>7} {'4-AUC':>7} {'3-AUC':>7} {'2-AUC':>7}")
print('=' * 80)
print(f"{'LSTM distilled T2a0.3 (wearable only)':<50} {'37.6%':>7} {'0.6879':>7} {'0.6858':>7} {'0.6846':>7}")
print(f"{'Hybrid LSTM T2a0.5 (wearable+survey)':<50} {'43.3%':>7} {'0.7167':>7} {'0.7213':>7} {'0.7474':>7}")
print(f"{'Teacher (wearable+CGM)':<50} {'45.4%':>7} {'0.7472':>7} {'—':>7} {'—':>7}")
print('-' * 80)

# LightGBM results
print(f"{'LGB 4-class (wearable+survey)':<50} {acc4*100:>6.1f}% {auc4_4:>7.4f} {auc3_4:>7.4f} {auc2_4:>7.4f}")
print(f"{'LGB 3-class dedicated':<50} {'—':>7} {'—':>7} {auc3_dedicated:>7.4f} {'—':>7}")
print(f"{'LGB 2-class dedicated':<50} {'—':>7} {'—':>7} {'—':>7} {auc2_dedicated:>7.4f}")

print(f'\nImprovement vs prev best (0.6879):')
print(f'  LGB 4-class:  {auc4_4 - 0.6879:+.4f}')
print(f'  LGB 3-class:  {auc3_dedicated - 0.6858:+.4f} (vs 0.6858)')
print(f'  LGB 2-class:  {auc2_dedicated - 0.6846:+.4f} (vs 0.6846)')

Model                                                  Acc   4-AUC   3-AUC   2-AUC
LSTM distilled T2a0.3 (wearable only)                37.6%  0.6879  0.6858  0.6846
Hybrid LSTM T2a0.5 (wearable+survey)                 43.3%  0.7167  0.7213  0.7474
Teacher (wearable+CGM)                               45.4%  0.7472       —       —
--------------------------------------------------------------------------------
LGB 4-class (wearable+survey)                        46.3%  0.7024  0.7171  0.7658
LGB 3-class dedicated                                    —       —  0.7321       —
LGB 2-class dedicated                                    —       —       —  0.7718

Improvement vs prev best (0.6879):
  LGB 4-class:  +0.0145
  LGB 3-class:  +0.0463 (vs 0.6858)
  LGB 2-class:  +0.0872 (vs 0.6846)
